# 04_sentiment_analysis (v2) — Sentiment5 (Target-based) RoBERTa

Goal:
- Run Sentiment-5 inference using `cardiffnlp/twitter-roberta-base-topic-sentiment-latest`
- Output per-thread sentiment fields required by Module 05/07:
  - sent5_label
  - sent5_probs (length=5)
  - sent5_valence_expected  (Σ probs * [-2,-1,0,1,2])
  - sent5_extremity         (abs(expected))
  - sent5_uncertainty_entropy
  - sent5_target

Input (Data-only):
- Data/data_03_cleaned_comments_*/cleaned_top_level_comments.parquet
- optional: cleaned_replies.parquet (if you decide to run sentiment on replies)

Output (Data-only):
- Data/data_04_sentiment5_YYYYMMDD_HHMMSS/


In [ ]:
# 📌 Cell 1 - Imports
# ------------------------------------------
import os
import glob
import json
import math
from datetime import datetime, timezone
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
# 📌 Cell 2 - Locate Data-only Inputs (latest data_03_cleaned_comments_*)
# ----------------------------------------------------------------------
# ✅ Rule:
# 1) Only look under Data/
# 2) Prefer pointer file Data/LATEST_DATA_03.txt if exists
# 3) Otherwise pick the newest folder matching data_03_cleaned_comments_*

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT = os.path.join(PROJECT_ROOT, "Data")

PTR_03 = os.path.join(DATA_ROOT, "LATEST_DATA_03.txt")

def pick_latest_data03() -> str:
    if os.path.exists(PTR_03):
        with open(PTR_03, "r", encoding="utf-8") as f:
            path = f.read().strip()
        if path and os.path.exists(path):
            return path

    candidates = sorted(
        glob.glob(os.path.join(DATA_ROOT, "data_03_cleaned_comments_*")),
        reverse=True
    )
    if not candidates:
        raise FileNotFoundError(
            f"❌ Cannot find any data_03_cleaned_comments_* under {DATA_ROOT}. "
            "Please run Module 03 first."
        )
    return candidates[0]

DATA_03_DIR = pick_latest_data03()
TOP_CLEAN_PATH = os.path.join(DATA_03_DIR, "cleaned_top_level_comments.parquet")
REP_CLEAN_PATH = os.path.join(DATA_03_DIR, "cleaned_replies.parquet")  # optional

if not os.path.exists(TOP_CLEAN_PATH):
    raise FileNotFoundError(f"❌ Missing: {TOP_CLEAN_PATH}")

print("✅ Using DATA_03_DIR:", DATA_03_DIR)
print("✅ TOP_CLEAN_PATH:", TOP_CLEAN_PATH)
print("✅ REP_CLEAN_PATH exists:", os.path.exists(REP_CLEAN_PATH))


In [ ]:
# 📌 Cell 3 - Create output folder (Data-only)
# -------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(DATA_ROOT, f"data_04_sentiment5_{timestamp}")
os.makedirs(RUN_DIR, exist_ok=True)

OUT_TOP_SENT = os.path.join(RUN_DIR, "sentiment5_top_level.parquet")
OUT_REP_SENT = os.path.join(RUN_DIR, "sentiment5_replies.parquet")          # optional
OUT_REPORT   = os.path.join(RUN_DIR, "sentiment5_report.json")
OUT_PTR      = os.path.join(DATA_ROOT, "LATEST_DATA_04.txt")               # optional pointer

print("✅ RUN_DIR:", RUN_DIR)
print("✅ OUT_TOP_SENT:", OUT_TOP_SENT)
print("✅ OUT_REPORT:", OUT_REPORT)


In [ ]:
# 📌 Cell 4 - Load cleaned tables
# ------------------------------
top = pd.read_parquet(TOP_CLEAN_PATH)
print("✅ top shape:", top.shape)

# Optional: run sentiment for replies too
RUN_REPLIES_SENTIMENT = False  # ✅ default False (save time). Set True if needed.

rep = None
if RUN_REPLIES_SENTIMENT:
    if not os.path.exists(REP_CLEAN_PATH):
        raise FileNotFoundError("❌ replies parquet not found. Please run Module 03 cleaning for replies.")
    rep = pd.read_parquet(REP_CLEAN_PATH)
    print("✅ replies shape:", rep.shape)

top.head(2)


In [ ]:
# 📌 Cell 5 - Load local model (offline-friendly)
# ----------------------------------------------
# You should have downloaded models in Module 00.
# Please set MODEL_DIR to your local path under PROJECT_ROOT/models/...
#
# Example:
#   models/sentiment5_roberta_topic/
# or:
#   models/twitter-roberta-base-topic-sentiment-latest/
#
# ✅ If you only have HF online access, you can replace MODEL_DIR with model_id,
# but for your project we recommend local.

MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "sentiment5_roberta_topic")  # <-- change if your folder name differs
MODEL_ID  = "cardiffnlp/twitter-roberta-base-topic-sentiment-latest"

if not os.path.exists(MODEL_DIR):
    print("⚠️ Local MODEL_DIR not found:", MODEL_DIR)
    print("   You can either:")
    print("   (A) Fix MODEL_DIR to your local downloaded folder")
    print("   (B) Temporarily use online loading from MODEL_ID (requires internet)")
    USE_ONLINE = True
else:
    USE_ONLINE = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ device:", device)

load_source = MODEL_ID if USE_ONLINE else MODEL_DIR
tokenizer = AutoTokenizer.from_pretrained(load_source)
model = AutoModelForSequenceClassification.from_pretrained(load_source)
model.to(device)
model.eval()

# label list (model config)
id2label = model.config.id2label
label_list = [id2label[i] for i in range(len(id2label))]
print("✅ labels:", label_list)


In [ ]:
# 📌 Cell 6 - Target strategy (IMPORTANT)
# --------------------------------------
# This model expects:  text + " </s> " + target
#
# Since you are doing a 2×2 design, we need a stable rule:
# - shein_* groups: target = "SHEIN"
# - non_shein_* groups: target = "fast fashion"
#
# ✅ You can change targets later, but this rule is consistent and easy to explain in README.

DEFAULT_TARGET_SHEIN = "SHEIN"
DEFAULT_TARGET_NON_SHEIN = "fast fashion"

def choose_target(cell_id: str, brand: str = None) -> str:
    c = (cell_id or "").lower()
    if "shein" in c:
        return DEFAULT_TARGET_SHEIN
    return DEFAULT_TARGET_NON_SHEIN

# weight mapping for expected valence
# labels: strongly negative / negative / negative or neutral / positive / strongly positive
VALENCE_WEIGHTS = np.array([-2, -1, 0, 1, 2], dtype=np.float32)


In [ ]:
# 📌 Cell 7 - Inference helper
# ----------------------------
# Output fields:
# - sent5_label
# - sent5_probs (list of 5 floats)
# - sent5_valence_expected
# - sent5_extremity
# - sent5_uncertainty_entropy
# - sent5_target

def softmax_np(logits: np.ndarray) -> np.ndarray:
    m = logits.max(axis=-1, keepdims=True)
    ex = np.exp(logits - m)
    return ex / ex.sum(axis=-1, keepdims=True)

def entropy_np(probs: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    p = np.clip(probs, eps, 1.0)
    return -(p * np.log(p)).sum(axis=-1)

@torch.no_grad()
def run_sentiment5(
    df: pd.DataFrame,
    text_col: str = "text_model",
    cell_col: str = "cell_id",
    brand_col: str = "brand",
    batch_size: int = 64,
    max_length: int = 256
) -> pd.DataFrame:
    """
    df: input table (top-level or replies)
    text_col: cleaned text column, usually text_model
    """
    texts = df[text_col].astype(str).fillna("").tolist()
    cell_ids = df[cell_col].astype(str).fillna("").tolist()
    brands = df[brand_col].astype(str).fillna("").tolist() if brand_col in df.columns else [""] * len(df)

    targets = [choose_target(cell_ids[i], brands[i]) for i in range(len(df))]
    inputs = [f"{texts[i]} </s> {targets[i]}" for i in range(len(df))]

    all_probs = []
    all_labels = []
    all_expected = []
    all_extreme = []
    all_entropy = []

    for i in tqdm(range(0, len(inputs), batch_size), desc="Sentiment5 Inference"):
        batch_inputs = inputs[i:i+batch_size]
        enc = tokenizer(
            batch_inputs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits.detach().cpu().numpy()
        probs = softmax_np(logits)

        # label
        pred_idx = probs.argmax(axis=-1)
        pred_label = [id2label[int(j)] for j in pred_idx]

        # expected valence & extremity
        expected = (probs * VALENCE_WEIGHTS.reshape(1, -1)).sum(axis=-1)
        extreme = np.abs(expected)

        # uncertainty
        ent = entropy_np(probs)

        all_probs.extend(probs.tolist())
        all_labels.extend(pred_label)
        all_expected.extend(expected.tolist())
        all_extreme.extend(extreme.tolist())
        all_entropy.extend(ent.tolist())

    out = df.copy()
    out["sent5_target"] = targets
    out["sent5_label"] = all_labels
    out["sent5_probs"] = all_probs
    out["sent5_valence_expected"] = all_expected
    out["sent5_extremity"] = all_extreme
    out["sent5_uncertainty_entropy"] = all_entropy

    return out


In [ ]:
# 📌 Cell 8 - Run sentiment on Top-level comments
# -----------------------------------------------
# ✅ Batch size recommendation:
# Ultra 5 225H CPU: start with 64. If OOM, reduce to 32.

BATCH_SIZE = 64
MAX_LEN = 256

# Ensure required columns exist
if "text_model" not in top.columns:
    raise ValueError("❌ top table missing text_model. Please run Module 03 v2 cleaning first.")

top_out = run_sentiment5(
    top,
    text_col="text_model",
    cell_col="cell_id",
    brand_col="brand" if "brand" in top.columns else None,
    batch_size=BATCH_SIZE,
    max_length=MAX_LEN
)

top_out.head(2)


In [ ]:
# 📌 Cell 9 - Optional: Run sentiment on replies
# ---------------------------------------------
rep_out = None
if RUN_REPLIES_SENTIMENT:
    if "text_model" not in rep.columns:
        raise ValueError("❌ replies table missing text_model. Please run Module 03 cleaning for replies first.")

    rep_out = run_sentiment5(
        rep,
        text_col="text_model",
        cell_col="cell_id",
        brand_col="brand" if (rep is not None and "brand" in rep.columns) else None,
        batch_size=BATCH_SIZE,
        max_length=MAX_LEN
    )
    rep_out.head(2)
else:
    print("ℹ️ RUN_REPLIES_SENTIMENT=False: skip replies sentiment (recommended for now).")


In [ ]:
# 📌 Cell 10 - Build sentiment report (Quality Gate style)
# -------------------------------------------------------
def summarize_sentiment(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    g = df.groupby("cell_id").agg(
        n=("comment_id", "count"),
        mean_valence=("sent5_valence_expected", "mean"),
        mean_extremity=("sent5_extremity", "mean"),
        mean_entropy=("sent5_uncertainty_entropy", "mean"),
        strong_neg_rate=("sent5_label", lambda s: (s=="strongly negative").mean()),
        neg_rate=("sent5_label", lambda s: ((s=="negative") | (s=="strongly negative")).mean()),
        pos_rate=("sent5_label", lambda s: ((s=="positive") | (s=="strongly positive")).mean()),
    ).reset_index()

    # label distribution
    dist = (
        df.groupby(["cell_id", "sent5_label"])
          .size()
          .reset_index(name="count")
    )
    return {
        "name": name,
        "by_cell_summary": g.to_dict(orient="records"),
        "label_distribution": dist.to_dict(orient="records")
    }

report: Dict[str, Any] = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_dir": RUN_DIR,
    "model": {
        "model_id": MODEL_ID,
        "load_source": (MODEL_ID if USE_ONLINE else MODEL_DIR),
        "labels": label_list,
        "valence_weights": VALENCE_WEIGHTS.tolist(),
        "max_length": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "device": device
    },
    "inputs": {
        "data_03_dir": DATA_03_DIR,
        "top_clean_path": TOP_CLEAN_PATH,
        "replies_clean_path": REP_CLEAN_PATH if RUN_REPLIES_SENTIMENT else None
    },
    "outputs": {
        "sentiment5_top_level": OUT_TOP_SENT,
        "sentiment5_replies": OUT_REP_SENT if RUN_REPLIES_SENTIMENT else None,
        "report_json": OUT_REPORT
    },
    "summary": {
        "top_level": summarize_sentiment(top_out, "top_level"),
        "replies": summarize_sentiment(rep_out, "replies") if (rep_out is not None) else None
    }
}

print("\n" + "="*50)
print("🚦 QUALITY GATE PASSED: SENTIMENT5 OUTPUT READY")
print("="*50)

# Quick view: shein_labor vs non_shein_labor (if present)
print("📊 Per-cell quick summary (top-level):")
df_summary = pd.DataFrame(report["summary"]["top_level"]["by_cell_summary"])
display(df_summary)


In [ ]:
# 📌 Cell 11 - Save outputs (Data-only)
# ------------------------------------
top_out.to_parquet(OUT_TOP_SENT, index=False)
print("✅ Saved:", OUT_TOP_SENT)

if rep_out is not None:
    rep_out.to_parquet(OUT_REP_SENT, index=False)
    print("✅ Saved:", OUT_REP_SENT)

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print("✅ Saved:", OUT_REPORT)

# Optional pointer
with open(OUT_PTR, "w", encoding="utf-8") as f:
    f.write(RUN_DIR)
print("✅ Updated pointer:", OUT_PTR)


In [ ]:
# 📌 Cell 12 - Minimal sanity checks for NLI/topic
# -----------------------------------------------
# For 07 negative-parent-only pipeline, we want enough negative parents.

neg_mask = top_out["sent5_label"].isin(["negative", "strongly negative"])
print("Top-level negative parents:", int(neg_mask.sum()), "/", len(top_out), f"({neg_mask.mean()*100:.1f}%)")

by_cell_neg = (
    top_out.assign(is_neg=neg_mask.astype(int))
           .groupby("cell_id")["is_neg"]
           .agg(["count", "sum", "mean"])
           .reset_index()
)
display(by_cell_neg)
